# 08 — Colab GPU Validation for Mamba-from-Scratch

Use this notebook in **Google Colab with a GPU runtime** to run the repo end-to-end on CUDA, generate benchmark artifacts, and download the results.

**Before running:** `Runtime -> Change runtime type -> T4 / GPU` (or any CUDA GPU Colab gives you today).

## What this notebook does

1. clones the public repo
2. installs dependencies for Colab
3. verifies CUDA is available
4. runs the full GPU validation script
5. previews the generated JSON results and figures
6. zips artifacts so you can download them back to your machine


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/varad-more/mamba-from-scratch.git"
REPO_DIR = Path('/content/mamba-from-scratch')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')

%cd /content/mamba-from-scratch
!git pull --ff-only || true


In [ ]:
# Install Colab-friendly dependencies.
# If Colab already has a suitable torch build, this will usually be quick/no-op-ish.
%pip install -q --upgrade pip
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -q -e .[dev,bench,kernel]


In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA device count:', torch.cuda.device_count())
else:
    raise RuntimeError('CUDA is not available. Switch the Colab runtime to GPU and rerun.')


In [ ]:
# Optional sanity check.
!pytest -q


## Run the full GPU validation bundle

You can tweak the flags below if Colab gives you a smaller GPU or you want a faster smoke test.

In [ ]:
PARITY_LAYERS = '0,5,23'
BATCH = 2
CHANNELS = 64
STATE = 16
LENGTH = 256
WARMUP = 5
REPEATS = 20
NEW_TOKENS = 16

!python scripts/run_gpu_validation.py \
  --device auto \
  --batch {BATCH} \
  --channels {CHANNELS} \
  --state {STATE} \
  --length {LENGTH} \
  --warmup {WARMUP} \
  --repeats {REPEATS} \
  --new-tokens {NEW_TOKENS} \
  --parity-layers {PARITY_LAYERS}


## Inspect generated JSON outputs

In [ ]:
import json
from pathlib import Path

results_dir = Path('benchmarks/results')
for path in sorted(results_dir.glob('*.gpu.json')):
    print(f'\n=== {path.name} ===')
    payload = json.loads(path.read_text())
    if isinstance(payload, list):
        for row in payload:
            print(row)
    else:
        print(json.dumps(payload, indent=2)[:4000])


## Preview generated figures

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figure_paths = [
    Path('figures/scan_benchmark_gpu.png'),
    Path('figures/inference_comparison_gpu.png'),
    Path('figures/roofline.png'),
]

for path in figure_paths:
    if path.exists():
        print(f'\n--- {path} ---')
        display(Image(filename=str(path)))
    else:
        print(f'Missing: {path}')


## Download artifacts from Colab

In [ ]:
import shutil
from google.colab import files

archive_base = '/content/mamba_gpu_validation_artifacts'
archive_path = shutil.make_archive(archive_base, 'zip', root_dir='.', base_dir='benchmarks/results')
print('Created:', archive_path)
files.download(archive_path)


## If something breaks

Common chaos sources:
- Colab gave you a CPU runtime instead of GPU
- Triton / CUDA wheel mismatch
- model download/network hiccup
- Colab memory constraints on a weaker GPU

If that happens, rerun the failing cell and send the traceback/logs.